# Reviewer-safe 13-experiment EEGMMIDB batch
Four capacity points, four neural baselines, three unweighted 83 M seeds, and three effective-number-weighted 83 M seeds. Grouped inner-validation subjects select checkpoints; untouched outer-test subjects are evaluated once.

In [ ]:
import os, shutil, subprocess, sys, zipfile
from pathlib import Path
subprocess.run(['nvidia-smi'], check=False)
runner_input = subprocess.check_output("find /kaggle/input -path '*/src/run_complete_batch.py' | head -1", shell=True, text=True).strip()
work = Path('/kaggle/working/eegmi-mstcnn-reproducibility')
if work.exists(): shutil.rmtree(work)
if runner_input:
    shutil.copytree(Path(runner_input).parents[1], work)
else:
    subprocess.run(['git', 'clone', '--depth', '1', 'https://github.com/Ronitreddy10/eegmi-mstcnn-reproducibility.git', str(work)], check=True)
edf_files = list(Path('/kaggle/input').rglob('*.edf'))
print('Attached EDF files:', len(edf_files))
if len(edf_files) < 600:
    raise FileNotFoundError('Complete EEGMMIDB input is not attached; expected at least 600 EDF files.')
edf_root = Path(os.path.commonpath([str(p.parent) for p in edf_files]))
os.environ['EEGMMIDB_SOURCE_DIR'] = str(edf_root)
print('Code package:', work)
print('EEGMMIDB source:', os.environ['EEGMMIDB_SOURCE_DIR'])


In [ ]:
# Do not alter these manuscript-run settings.
MAX_WALL_HOURS = 8.0
GRID = work / 'configs' / 'reviewer_safe_13.json'
RESULTS = Path('/kaggle/working/reviewer_safe_13_results')
RESULTS.mkdir(parents=True, exist_ok=True)
previous = subprocess.check_output("find /kaggle/input -name 'reviewer_safe_13_results_export*.zip' | head -1", shell=True, text=True).strip()
previous_marker = subprocess.check_output("find /kaggle/input \( -name 'BATCH_STATUS.json' -o -name 'complete_batch_manifest.json' \) | head -1", shell=True, text=True).strip()
if previous:
    print('Resuming from ZIP:', previous)
    with zipfile.ZipFile(previous) as archive: archive.extractall(RESULTS)
elif previous_marker:
    previous_root = Path(previous_marker).parent
    print('Resuming from extracted results:', previous_root)
    for item in previous_root.iterdir():
        target = RESULTS / item.name
        if item.is_dir(): shutil.copytree(item, target, dirs_exist_ok=True)
        else: shutil.copy2(item, target)
cmd = [sys.executable, str(work/'src'/'run_complete_batch.py'), '--grid', str(GRID), '--data-dir', '/kaggle/working/eeg_data', '--output-root', str(RESULTS), '--checkpoint-archive', '/kaggle/working/reviewer_safe_13_results_export', '--max-wall-hours', str(MAX_WALL_HOURS), '--epochs', '60', '--patience', '8', '--batch-size', '64', '--refit-mode', 'none']
print(' '.join(cmd), flush=True)
subprocess.run(cmd, check=True)


In [ ]:
summaries = list(RESULTS.glob('*/summary.json'))
if summaries:
    subprocess.run([sys.executable, str(work/'src'/'summarize_grid.py'), '--results', str(RESULTS)], check=True)
archive = shutil.make_archive('/kaggle/working/reviewer_safe_13_results_export', 'zip', root_dir=RESULTS)
status = RESULTS / 'BATCH_STATUS.json'
print(status.read_text() if status.exists() else 'Use the checkpoint archive to resume.')
print('DOWNLOAD:', archive)
